# Imbalance Handling:class weight & SMOTE

In [4]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

df = pd.read_csv("../data/creditcard.csv")

df["Hour"] = (df["Time"] // 3600) % 24
df["Amount"] = np.log1p(df["Amount"])   

X = df.drop(columns=["Class", "Time"])
y = df["Class"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

model = LogisticRegression(max_iter=1000, random_state=42)
model.fit(X_train, y_train)

accuracy = (model.predict(X_test) == y_test).mean()  
print(f"Accuracy: {accuracy:.4f}")

Accuracy: 0.9992


In [5]:
from sklearn.metrics import confusion_matrix, classification_report

lr_weighted = LogisticRegression(
    max_iter=1000, class_weight="balanced", random_state=42
)
lr_weighted.fit(X_train, y_train)

y_pred_w = lr_weighted.predict(X_test)
print(confusion_matrix(y_test, y_pred_w))
print(classification_report(y_test, y_pred_w, digits=4))

accuracy = (y_pred_w == y_test).mean()  
print(f"Accuracy: {accuracy:.4f}")

[[55347  1517]
 [    9    89]]
              precision    recall  f1-score   support

           0     0.9998    0.9733    0.9864     56864
           1     0.0554    0.9082    0.1045        98

    accuracy                         0.9732     56962
   macro avg     0.5276    0.9407    0.5454     56962
weighted avg     0.9982    0.9732    0.9849     56962

Accuracy: 0.9732


## Class weights: recall up, precision collapses

Setting `class_weight="balanced"` makes each fraud error count ~578× more
than a normal-transaction error (the 284,315 / 492 ratio), pushing the model
from conservative to aggressive.

| Model             | Recall (fraud) | Precision (fraud) | FP    | Missed frauds (FN) |
|-------------------|:--------------:|:-----------------:|:-----:|:------------------:|
| Baseline LR       | 0.65           | 0.83              | 13    | 34                 |
| LR class_weight   | 0.91           | 0.055             | 1,517 | 9                  |

Recall jumps from 0.65 to 0.91 — only 9 of 98 frauds now slip through, down
from 34. But precision collapses from 0.83 to 0.055: false positives explode
from 13 to 1,517, roughly 17 false alarms per real fraud caught. Only 2.7% of
normal transactions were misflagged, but 2.7% of 56,864 is still thousands —
the small-percentage-of-a-huge-denominator effect again.

Note that overall accuracy *dropped* from 0.9992 to 0.9732 while the model
became far better at its actual job — a final nail in accuracy's coffin for
this problem.

**Is this "better"?** It depends on the cost of a false positive. If each flag
triggers an automated "approve this transaction?" SMS, then 1,517 mildly
annoyed customers may be an acceptable price for catching 25 extra frauds. If
each flag means manual review by an analyst, it is not. Mathematics does not
decide this — which is why we are going to use threshold tuning to find intermediate
operating points instead of choosing between two extremes.

## SMOTE
SMOTE (Synthetic Minority Over-sampling Technique) does not simply duplicate
existing frauds. It generates *synthetic* samples by interpolation: it takes a
fraud sample, finds its nearest neighbours among the other frauds, and creates
new points along the line segments connecting them in the 30-dimensional
feature space.

Crucially, SMOTE is applied to the **training set only**. The test set keeps
its original 98 real frauds untouched, so evaluation still reflects real-world
performance.

Resampling before the split leaks information, because synthetic test points are interpolated from training samples — always split first, resample only the training set.

In [7]:
from imblearn.over_sampling import SMOTE

smote = SMOTE(random_state=42)
X_train_sm, y_train_sm = smote.fit_resample(X_train, y_train)

print(f"Before:  {y_train.value_counts().to_dict()}")
print(f"After: {y_train_sm.value_counts().to_dict()}")

lr_smote = LogisticRegression(max_iter=1000, random_state=42)
lr_smote.fit(X_train_sm, y_train_sm)

y_pred_sm = lr_smote.predict(X_test)
print(confusion_matrix(y_test, y_pred_sm))
print(classification_report(y_test, y_pred_sm, digits=4))

Before:  {0: 227451, 1: 394}
After: {0: 227451, 1: 227451}
[[55288  1576]
 [    8    90]]
              precision    recall  f1-score   support

           0     0.9999    0.9723    0.9859     56864
           1     0.0540    0.9184    0.1020        98

    accuracy                         0.9722     56962
   macro avg     0.5269    0.9453    0.5440     56962
weighted avg     0.9982    0.9722    0.9844     56962



## Class weights vs SMOTE: equivalent results

Both approaches produced almost identical results (recall ≈ 0.91–0.92,
precision ≈ 0.054–0.055). This is expected: both do the same thing by
different means — they force the model to treat the two classes as equally
important. `class_weight="balanced"` weights the few frauds ~578× more heavily
in the loss function; SMOTE instead oversamples the frauds until the classes
are balanced (~394 → 227,451). For a linear model like Logistic Regression,
both routes push the decision boundary to nearly the same place.

**Choice:** `class_weight` is preferable here — it achieves the same result
without roughly doubling the training set size and training time. SMOTE's
advantage typically appears with non-linear models and more complex
distributions, which is not the case here.

## Threshold Tuning

In [8]:
from sklearn.metrics import precision_recall_curve

y_scores = model.predict_proba(X_test)[:, 1]

precision, recall, thresholds = precision_recall_curve(y_test, y_scores)

import numpy as np
precision, recall = precision[:-1], recall[:-1]  
mask = recall >= 0.85
best = np.argmax(np.where(mask, precision, -1))

print(f"Threshold: {thresholds[best]:.4f}")
print(f"Recall:    {recall[best]:.4f}")
print(f"Precision: {precision[best]:.4f}")

Threshold: 0.0239
Recall:    0.8776
Precision: 0.5584


In [9]:
y_pred_tuned = (y_scores >= 0.0239).astype(int)
print(confusion_matrix(y_test, y_pred_tuned))
print(classification_report(y_test, y_pred_tuned, digits=4))

[[56796    68]
 [   13    85]]
              precision    recall  f1-score   support

           0     0.9998    0.9988    0.9993     56864
           1     0.5556    0.8673    0.6773        98

    accuracy                         0.9986     56962
   macro avg     0.7777    0.9331    0.8383     56962
weighted avg     0.9990    0.9986    0.9987     56962



## Threshold tuning beats resampling

Taking the original baseline model and simply lowering the decision threshold
to 0.024 (target: recall ≥ 0.85) produced a dramatically better operating
point than either class weights or SMOTE:

| Approach                     | Recall | Precision | FP    | FN | F1 (fraud) |
|------------------------------|:------:|:---------:|:-----:|:--:|:----------:|
| LR class_weight              | 0.91   | 0.055     | 1,517 | 9  | 0.10       |
| LR + SMOTE                   | 0.92   | 0.054     | 1,576 | 8  | 0.10       |
| **Baseline + tuned threshold** | 0.88 | **0.558** | **68** | 13 | **0.68**   |

For essentially the same recall, false positives dropped from ~1,500 to 68 —
about 22× fewer false alarms — and precision rose tenfold (0.055 → 0.558).

**Why:** class weights and SMOTE distort the model itself, forcing it to treat
frauds as if they were as common as normal transactions, which inflates its
probability estimates everywhere. Threshold tuning instead keeps the original,
undistorted model — whose probabilities reflect the true fraud likelihood —
and just moves the cut-off point. A well-calibrated model with a smart
threshold beats a distorted model at the default 0.5.

**Take-home:** in imbalanced problems, *where you set the threshold* often
matters more than *how you train*. A common junior mistake is to reach for
SMOTE and stop; here, plain threshold tuning on an unmodified model
outperforms it.